In [ ]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. CHARGEMENT
df = pd.read_gbq("SELECT * FROM `ton_projet.dataset_marts.mart_analysis_full`")

# 2. TEST DE NORMALITÉ (Shapiro-Wilk)
# Note: Sur de très grands échantillons, le test de Shapiro est très sensible.
# Si ton dataset est immense (>5000 lignes), utilise le test de Kolmogorov-Smirnov.
stat, p_value_shapiro = stats.shapiro(df['price'])
print(f"--- Test de Shapiro-Wilk (Normalité) ---")
print(f"P-value: {p_value_shapiro:.4f}")
print("Interprétation: " + ("Distribution Normale" if p_value_shapiro > 0.05 else "Distribution Non-Normale"))

# 3. ANOVA (One-way)
# On compare les moyennes de prix entre les sources (Amazon, Jumia, Avito)
model = ols('price ~ C(source)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("\n--- Résultat ANOVA ---")
print(anova_table)

# 4. TEST POST-HOC (Tukey HSD)
# Si l'ANOVA est significative (p < 0.05), on identifie quelles plateformes diffèrent
if anova_table['PR(>F)'][0] < 0.05:
    print("\n--- Test Post-hoc (Tukey HSD) ---")
    tukey = pairwise_tukeyhsd(endog=df['price'], groups=df['source'], alpha=0.05)
    print(tukey)
else:
    print("\nL'ANOVA n'est pas significative, pas besoin de test post-hoc.")